In [ ]:
# ===========================================
# Fruits-360 FDCNN + Robust Score-CAM (FFT→Spatial Mapping)
# ===========================================
import os, warnings, math, gc
warnings.filterwarnings("ignore")

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft as fft
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from skimage import morphology
from scipy.ndimage import gaussian_filter, uniform_filter

from tqdm import tqdm

# ----------------- Device & Determinism -----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.benchmark = True   # speed
    torch.backends.cudnn.deterministic = False

# =========================================================
# 1) Dataset
# =========================================================
class FruitsDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root = root_dir
        self.t = transform
        self.classes = sorted([d for d in os.listdir(root_dir)
                               if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {c:i for i,c in enumerate(self.classes)}
        self.samples = []
        for c in self.classes:
            p = os.path.join(root_dir, c)
            for f in os.listdir(p):
                if f.lower().endswith((".png",".jpg",".jpeg")):
                    self.samples.append((os.path.join(p,f), self.class_to_idx[c]))
        print(f"[FruitsDataset] {len(self.samples)} images, {len(self.classes)} classes")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, y = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.t: img = self.t(img)
        return img, y

def load_fruits_dataset(data_root):
    # Fruits-360 100x100, keep transforms simple and stable
    normalize = transforms.Normalize(mean=[0.485,0.456,0.406],
                                     std=[0.229,0.224,0.225])
    t_train = transforms.Compose([
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        normalize
    ])
    t_test = transforms.Compose([
        transforms.ToTensor(),
        normalize
    ])
    train = FruitsDataset(os.path.join(data_root,"Training"), t_train)
    test  = FruitsDataset(os.path.join(data_root,"Test"), t_test)
    return train, test, train.classes

# =========================================================
# 2) Spatial <-> Frequency helpers
# =========================================================
def spatial_to_frequency(img_batch):
    """
    img_batch: (B,3,H,W) normalized
    returns: freq_features (B,9,H,W), phase (B,3,H,W), freq_complex (B,3,H,W)
    """
    # FFT per-channel
    f = fft.fft2(img_batch, dim=(-2,-1))
    f = fft.fftshift(f, dim=(-2,-1))
    mag = torch.abs(f) + 1e-8
    phase = torch.angle(f)

    # robust log-norm per image/channel
    mag_log = torch.log(mag)
    # normalize per-sample to keep dynamic range consistent across fruits w/ white bg
    mean = mag_log.mean(dim=(-2,-1), keepdim=True)
    std  = mag_log.std(dim=(-2,-1), keepdim=True).clamp_min(1e-6)
    mag_norm = (mag_log - mean)/std

    cos_p = torch.cos(phase)
    sin_p = torch.sin(phase)
    feat = torch.cat([mag_norm, cos_p, sin_p], dim=1)  # (B,9,H,W)
    return feat, phase, f

def denorm_image(x):
    """x: (3,H,W) normalized -> (H,W,3) in [0,1]"""
    mean = torch.tensor([0.485,0.456,0.406], device=x.device).view(3,1,1)
    std  = torch.tensor([0.229,0.224,0.225], device=x.device).view(3,1,1)
    y = (x*std + mean).clamp(0,1)
    return y.permute(1,2,0).detach().cpu().numpy()

# =========================================================
# 3) Frequency-Domain Model (ResNet50)
# =========================================================
class FrequencyDomainCNN(nn.Module):
    def __init__(self, num_classes, dropout=0.4):
        super().__init__()
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        m.conv1 = nn.Conv2d(9, 64, kernel_size=7, stride=2, padding=3, bias=False)
        nf = m.fc.in_features
        m.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(nf, 1024), nn.ReLU(inplace=True), nn.BatchNorm1d(1024),
            nn.Dropout(dropout),
            nn.Linear(1024, num_classes)
        )
        self.model = m
        # init new conv weights
        nn.init.kaiming_normal_(self.model.conv1.weight, mode="fan_out", nonlinearity="relu")

    def forward(self,x): return self.model(x)

    @torch.no_grad()
    def get_activations(self, x):
        # forward till last conv block
        x = self.model.conv1(x); x = self.model.bn1(x); x = self.model.relu(x); x = self.model.maxpool(x)
        x = self.model.layer1(x); x = self.model.layer2(x); x = self.model.layer3(x); x = self.model.layer4(x)
        return x  # (B,2048,h,w)

# =========================================================
# 4) Train utils (fast, stable)
# =========================================================
class EarlyStop:
    def __init__(self, patience=10, min_delta=0.0):
        self.patience, self.min_delta = patience, min_delta
        self.best, self.count, self.state = -1, 0, None
    def step(self, metric, model):
        if self.best < 0 or metric > self.best + self.min_delta:
            self.best, self.count = metric, 0
            self.state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            self.count += 1
        return self.count >= self.patience

def train_model(model, train_loader, val_loader, epochs=25, base_lr=1e-3, wd=5e-4):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    # different LR for new layers
    pre_params, new_params = [], []
    for n,p in model.named_parameters():
        (new_params if ('fc' in n or 'conv1' in n) else pre_params).append(p)
    opt = torch.optim.AdamW([
        {'params': pre_params, 'lr': base_lr*0.1},
        {'params': new_params, 'lr': base_lr}
    ], weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type=='cuda'))
    stopper = EarlyStop(patience=8, min_delta=0.2)

    tr_losses, va_losses, tr_accs, va_accs = [], [], [], []
    for ep in range(1, epochs+1):
        # ---- train
        model.train()
        running, correct, total = 0.0, 0, 0
        pbar = tqdm(train_loader, desc=f"Epoch {ep}/{epochs} [Train]", leave=False)
        for x,y,_ in pbar:
            x,y = x.to(device,non_blocking=True), y.to(device,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
                out = model(x)
                loss = criterion(out,y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()

            running += loss.item()*y.size(0)
            pred = out.argmax(1)
            total += y.size(0); correct += (pred==y).sum().item()
            pbar.set_postfix(loss=f"{loss.item():.3f}", acc=f"{100*correct/total:.2f}%")
        tr_loss = running/total; tr_acc = 100*correct/total
        tr_losses.append(tr_loss); tr_accs.append(tr_acc)

        # ---- val
        model.eval()
        v_running, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for x,y,_ in val_loader:
                x,y = x.to(device,non_blocking=True), y.to(device,non_blocking=True)
                out = model(x); loss = criterion(out,y)
                v_running += loss.item()*y.size(0)
                v_total += y.size(0); v_correct += (out.argmax(1)==y).sum().item()
        va_loss = v_running/v_total; va_acc = 100*v_correct/v_total
        va_losses.append(va_loss); va_accs.append(va_acc)
        sched.step()

        print(f"Epoch {ep:02d}: train loss {tr_loss:.3f} acc {tr_acc:.2f}% | val loss {va_loss:.3f} acc {va_acc:.2f}%")

        if stopper.step(va_acc, model):
            print("Early stopping.")
            break

    if stopper.state is not None:
        model.load_state_dict(stopper.state, strict=True)
        print(f"Loaded best model (val acc {stopper.best:.2f}%).")
    return tr_losses, va_losses, tr_accs, va_accs

# =========================================================
# 5) Robust Score-CAM (with strong weighting) + FFT→Spatial projection
# =========================================================
class RobustScoreCAM:
    """
    More stable than vanilla Score-CAM:
      - uses temperature-scaled softmax on logits
      - strong per-map normalization + sigmoid sharpening
      - weight = max(0, s_masked - s_base*0.05) with floor to avoid flat maps
      - channel sampling if too many maps (speed)
    """
    def __init__(self, model, max_maps=1024, temperature=2.0):
        self.model = model.eval()
        self.max_maps = max_maps
        self.temperature = temperature

    @torch.no_grad()
    def generate(self, x, target_class):
        acts = self.model.get_activations(x)            # (1,K,h,w)
        _,K,h,w = acts.shape
        # sample high-variance maps if huge
        if K > self.max_maps:
            var = acts.view(K,-1).var(dim=1)
            topk = torch.topk(var, self.max_maps).indices
            acts = acts[:, topk]
            K = acts.size(1)

        # base score
        base = F.softmax(self.model(x)/self.temperature, dim=1)[0,target_class].item()

        # upsample to input
        H,W = x.shape[-2:]
        up = F.interpolate(acts, size=(H,W), mode="bilinear", align_corners=False)[0]  # (K,H,W)

        weights = []
        for k in range(K):
            m = up[k]
            m = m - m.min()
            if m.max() > 0: m = m/m.max()
            m = torch.sigmoid((m-0.5)*8.0)  # sharpen
            masked = x * m.unsqueeze(0).unsqueeze(0)     # broadcast to 9ch
            s = F.softmax(self.model(masked)/self.temperature, dim=1)[0,target_class].item()
            w = max(0.0, s - base*0.05)
            weights.append(w)

        w = torch.tensor(weights, device=x.device)
        if w.sum() > 0: w = F.softmax(w*12.0, dim=0)     # emphasize peaks
        cam = (w.view(-1,1,1)*acts[0]).sum(dim=0)        # (h,w)
        cam = F.relu(cam)
        cam = F.interpolate(cam.unsqueeze(0).unsqueeze(0), size=(H,W),
                            mode="bilinear", align_corners=False).squeeze()
        cam = cam / (cam.max()+1e-8)
        return cam.detach().cpu().numpy(), w.detach().cpu().numpy()

# ---------- FFT → Spatial projection (key fix for fruits) ----------
def cam_frequency_to_spatial_saliency(cam2d, original_img_tensor):
    """
    cam2d: (H,W) in [0,1] from Score-CAM over frequency features
    original_img_tensor: (3,H,W), normalized
    We project CAM to the spatial image by modulating the *magnitude* in FFT,
    iFFT back, and taking |enhanced - original| as a saliency estimate.
    """
    x = (original_img_tensor.clone()).unsqueeze(0)  # (1,3,H,W)
    # FFT of the original (not normalized features)
    Forig = fft.fftshift(fft.fft2(x, dim=(-2,-1)), dim=(-2,-1))  # (1,3,H,W)
    mag = torch.abs(Forig); phase = torch.angle(Forig)

    cam = torch.from_numpy(cam2d).to(x.device).float()
    if cam.shape != x.shape[-2:]:
        cam = F.interpolate(cam[None,None], size=x.shape[-2:], mode="bilinear", align_corners=False)[0,0]
    cam = (cam - cam.min())/(cam.max()-cam.min()+1e-8)

    # strengthen frequencies where CAM says "important"
    boost = 1.0 + 1.5*cam  # beta=1.5 works well on white backgrounds
    mag_enh = mag * boost.unsqueeze(0).unsqueeze(0)

    Fnew = mag_enh * torch.exp(1j*phase)
    Fnew = fft.ifftshift(Fnew, dim=(-2,-1))
    x_new = torch.real(fft.ifft2(Fnew, dim=(-2,-1)))  # (1,3,H,W)

    # saliency = per-pixel change magnitude
    sal = torch.abs(x_new - x).mean(dim=1)[0]  # (H,W)
    sal = (sal - sal.min())/(sal.max()-sal.min()+1e-8)
    sal = gaussian_filter(sal.detach().cpu().numpy(), sigma=1.0)

    # clean mask (remove tiny speckles on white bg)
    thr = np.percentile(sal, 70)
    mask = sal > thr
    mask = morphology.remove_small_objects(mask, min_size=40)
    mask = morphology.remove_small_holes(mask, area_threshold=40)
    mask = morphology.binary_dilation(mask, morphology.disk(2))
    sal = sal * mask.astype(np.float32)

    # guided smoothing w.r.t. image edges (simple box-guided filter)
    img_np = denorm_image(original_img_tensor)  # (H,W,3) in [0,1]
    guide = img_np.mean(axis=2)
    r, eps = 5, 1e-3
    mean_g = uniform_filter(guide, size=r)
    mean_s = uniform_filter(sal, size=r)
    corr_gs = uniform_filter(guide*sal, size=r)
    var_g = uniform_filter(guide*guide, size=r) - mean_g*mean_g
    a = (corr_gs - mean_g*mean_s) / (var_g + eps)
    b = mean_s - a*mean_g
    sal = uniform_filter(a, size=r)*guide + uniform_filter(b, size=r)
    sal = np.clip((sal - sal.min())/(sal.max()-sal.min()+1e-8), 0, 1)
    return sal

def color_overlay(img_np, saliency, alpha=0.5):
    cm = plt.cm.jet(saliency)[...,:3]
    a = gaussian_filter(saliency**0.8, sigma=1.0)[...,None]
    out = (1 - alpha*a)*img_np + alpha*a*cm
    return np.clip(out, 0, 1)

# =========================================================
# 6) Plotting (matches CIFAR-10 layout)
# =========================================================
def plot_scorecam_results(original_np, freq_magnitude_img, cam_freq, saliency_map,
                          highlighted, prediction, true_label, classes, confidence):
    fig, axes = plt.subplots(2,4, figsize=(20,10))
    fig.suptitle("Frequency Domain CNN - Score-CAM Explainability (Fruits-360)",
                 fontsize=16, fontweight="bold", y=0.995)

    # Original
    axes[0,0].imshow(original_np); axes[0,0].set_title(f"Original Image\nGround Truth: {classes[true_label]}", fontweight="bold"); axes[0,0].axis("off")

    # Frequency magnitude display
    im1 = axes[0,1].imshow(freq_magnitude_img, cmap="viridis"); axes[0,1].set_title("Frequency Domain\n(Magnitude Spectrum)", fontweight="bold"); axes[0,1].axis("off")
    plt.colorbar(im1, ax=axes[0,1], fraction=0.046, pad=0.04)

    # Score-CAM (frequency feature space, upsampled)
    im2 = axes[0,2].imshow(cam_freq, cmap="jet"); axes[0,2].set_title("Score-CAM\n(Frequency Feature Map)", fontweight="bold"); axes[0,2].axis("off")
    plt.colorbar(im2, ax=axes[0,2], fraction=0.046, pad=0.04)

    # Prediction box
    ok = (prediction==true_label)
    axes[0,3].text(0.5,0.5, f"{'✓' if ok else '✗'} Prediction:\n{classes[prediction]}\n\nConfidence:\n{confidence:.1f}%",
                   ha="center", va="center", fontsize=13, fontweight="bold",
                   bbox=dict(boxstyle="round", facecolor=("green" if ok else "red"), alpha=0.25))
    axes[0,3].set_title("Model Prediction", fontweight="bold"); axes[0,3].axis("off")

    # Saliency
    im3 = axes[1,0].imshow(saliency_map, cmap="hot"); axes[1,0].set_title("Saliency Map\n(FFT→Spatial Projection)", fontweight="bold"); axes[1,0].axis("off")
    plt.colorbar(im3, ax=axes[1,0], fraction=0.046, pad=0.04)

    # Highlighted
    axes[1,1].imshow(highlighted); axes[1,1].set_title("Highlighted Regions\n(Overlay)", fontweight="bold"); axes[1,1].axis("off")

    # Heatmap overlay
    axes[1,2].imshow(original_np); axes[1,2].imshow(saliency_map, cmap="jet", alpha=0.4); axes[1,2].set_title("Importance Heatmap\n(40% Overlay)", fontweight="bold"); axes[1,2].axis("off")

    # Before / After
    axes[1,3].imshow(np.concatenate([original_np, highlighted], axis=1)); axes[1,3].set_title("Before | After\n(Score-CAM Highlighting)", fontweight="bold"); axes[1,3].axis("off")

    plt.tight_layout()
    plt.show()

def plot_curves(tr_losses, va_losses, tr_accs, va_accs):
    fig, (a1,a2) = plt.subplots(1,2, figsize=(14,5))
    e = range(1, len(tr_losses)+1)
    a1.plot(e,tr_losses,label="Train"); a1.plot(e,va_losses,label="Val"); a1.set_title("Loss"); a1.legend(); a1.grid(True, alpha=0.3)
    a2.plot(e,tr_accs,label="Train"); a2.plot(e,va_accs,label="Val"); a2.set_title("Accuracy (%)"); a2.legend(); a2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

# =========================================================
# 7) Pipeline
# =========================================================
def main():
    print("="*80)
    print("Enhanced Frequency Domain CNN + Robust Score-CAM (Fruits-360)")
    print("="*80)

    # ---- path
    data_root = "/kaggle/input/fruits/fruits-360_100x100/fruits-360"  # <- change if needed

    # ---- data
    print("\n[1] Loading dataset ...")
    trainset, testset, classes = load_fruits_dataset(data_root)
    print(f"Classes: {len(classes)}")

    # split train -> train/val
    val_ratio = 0.15
    n = len(trainset); n_val = int(n*val_ratio)
    g = torch.Generator().manual_seed(42)
    train_idx, val_idx = torch.utils.data.random_split(range(n), [n-n_val, n_val], generator=g)
    train_sub = torch.utils.data.Subset(trainset, train_idx)
    val_sub   = torch.utils.data.Subset(trainset, val_idx)

    # wrap to frequency dataset
    class FreqDS(Dataset):
        def __init__(self, base): self.base=base
        def __len__(self): return len(self.base)
        def __getitem__(self,i):
            img,y = self.base[i]
            with torch.no_grad():
                f,p,_ = spatial_to_frequency(img.unsqueeze(0))
            return f.squeeze(0), y, p.squeeze(0)

    train_fd = FreqDS(train_sub); val_fd = FreqDS(val_sub); test_fd = FreqDS(testset)

    bs = 64
    train_loader = DataLoader(train_fd, batch_size=bs, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
    val_loader   = DataLoader(val_fd,   batch_size=bs, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)
    test_loader  = DataLoader(test_fd,  batch_size=1,  shuffle=False)

    # ---- model
    print("\n[2] Building model ...")
    model = FrequencyDomainCNN(num_classes=len(classes)).to(device)
    print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

    # ---- train
    print("\n[3] Training ...")
    trL, vaL, trA, vaA = train_model(model, train_loader, val_loader,
                                     epochs=10, base_lr=1e-3, wd=5e-4)
    print("\n[3.1] Curves")
    plot_curves(trL, vaL, trA, vaA)

    # ---- test
    print("\n[4] Testing ...")
    model.eval(); tot=0; ok=0
    with torch.no_grad():
        for x,y,_ in tqdm(test_loader, leave=False):
            x,y = x.to(device), y.to(device)
            out = model(x)
            ok += (out.argmax(1)==y).sum().item(); tot += y.size(0)
    test_acc = 100*ok/tot
    print(f"Final Test Accuracy: {test_acc:.2f}%")

    # ---- Explainability samples
    print("\n[5] Score-CAM + FFT→Spatial visualizations ...\n")
    np.random.seed(42)
    show_n = min(10, len(testset))
    indices = np.random.choice(len(testset), show_n, replace=False)

    scorer = RobustScoreCAM(model)

    for idx in indices:
        orig_img, true_y = testset[idx]           # (3,H,W) normalized
        freq, phase, _ = spatial_to_frequency(orig_img.unsqueeze(0))
        x_fd = freq.to(device)

        with torch.no_grad():
            logits = model(x_fd)
            prob = F.softmax(logits, dim=1)
            conf, pred = torch.max(prob, 1)
        pred_cls = pred.item(); confidence = float(conf.item()*100)

        # Score-CAM on frequency features
        cam_freq, _ = scorer.generate(x_fd, pred_cls)  # (H,W) in [0,1]

        # --- make a magnitude display just for the panel (averaged first 3 channels)
        freq_display = freq[:, :3].mean(1)[0].detach().cpu().numpy()

        # FFT→Spatial saliency using original image
        sal = cam_frequency_to_spatial_saliency(cam_freq, orig_img)
        overlay = color_overlay(denorm_image(orig_img), sal)

        # Plot like CIFAR panel
        plot_scorecam_results(
            denorm_image(orig_img),        # original_np
            freq_display,                  # frequency magnitude img
            cam_freq,                      # Score-CAM map (feature space)
            sal,                           # spatial saliency
            overlay,                       # highlighted image
            pred_cls, true_y, classes, confidence
        )

        if torch.cuda.is_available(): torch.cuda.empty_cache()
        gc.collect()

    # ---- save model
    print("\n[6] Saving model ...")
    torch.save({
        "model_state_dict": model.state_dict(),
        "classes": classes,
        "test_accuracy": test_acc
    }, "fruits_fdcnn_scorecam_best.pth")
    print("Saved: fruits_fdcnn_scorecam_best.pth")

    print("\nDone.")

if __name__ == "__main__":
    main()